## Part 5: All of This, Out of the Box with Contextual AI

Over the last two hours, we built every layer of an agentic RAG system by hand:
- Spun up a vector database and loaded data
- Built agents for dynamic filtering and query decomposition
- Wired up intent classification, slot filling, and multi-agent orchestration

Understanding **how** these layers work is critical. But when you need to ship a production system, [Contextual AI](https://contextual.ai) gives you all of this out of the box — managed infrastructure, built-in retrieval, agentic workflows, and grounded generation — so you can focus on the domain problem instead of the plumbing.

This notebook walks through the Contextual AI platform doing what we just built, in minutes.

## Setup

In [ ]:
#!pip install contextual-client python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from contextual import ContextualAI

load_dotenv()

API_KEY = os.getenv("CONTEXTUAL_API_KEY")

if not API_KEY:
    raise ValueError("Set CONTEXTUAL_API_KEY in your .env file")

client = ContextualAI(api_key=API_KEY)
print("Contextual AI client ready")

## What We Built vs. What Contextual AI Handles

| What we built manually | Contextual AI |
|---|---|
| Docker + Qdrant setup | Managed datastores — no infrastructure |
| Embedding + batch ingestion | Automatic parsing, chunking, and embedding |
| Dynamic filter agents | Built-in hybrid search with configurable retrieval |
| Query decomposition agents | Agentic research loops with multi-turn tool use |
| Intent classification + slot filling | Agent Composer workflows with conditional routing |
| Multi-agent orchestration | Agent Composer — visual or YAML-based orchestration |
| Manual evaluation (MRR, NDCG) | Built-in groundedness scores, feedback, and metrics API |

## Step 1: Create a Datastore

Instead of spinning up Docker and configuring a vector database, you create a managed datastore with one API call. Contextual AI handles storage, indexing, and embedding automatically.

In [ ]:
datastore_name = "ODSC_Workshop_Demo"

# Check if datastore already exists
datastores = client.datastores.list()
existing = next((ds for ds in datastores if ds.name == datastore_name), None)

if existing:
    datastore_id = existing.id
    print(f"Using existing datastore: {datastore_id}")
else:
    result = client.datastores.create(name=datastore_name)
    datastore_id = result.id
    print(f"Created datastore: {datastore_id}")

## Step 2: Ingest Documents

No manual embedding, no batch import loops, no vectorizer configuration. Upload your files and the platform handles parsing (including tables, figures, and complex layouts), chunking, and embedding.

In [ ]:
# Upload documents to the datastore
# Replace with your own files or use the sample data
import glob

data_files = glob.glob("../data/*.pdf") + glob.glob("../data/*.json")

document_ids = []
for file_path in data_files:
    try:
        with open(file_path, "rb") as f:
            result = client.datastores.documents.ingest(datastore_id, file=f)
            document_ids.append(result.id)
            print(f"Uploaded: {file_path} -> {result.id}")
    except Exception as e:
        print(f"Error uploading {file_path}: {e}")

print(f"\nUploaded {len(document_ids)} documents")

## Step 3: Create an Agent with Agent Composer

This is where all the manual orchestration we built — intent classification, slot filling, subagent routing, query decomposition — gets replaced by a single YAML configuration.

Agent Composer lets you define multi-step agentic workflows that include:
- **Hybrid search** with configurable semantic/lexical balance
- **Reranking** with state-of-the-art models
- **Agentic research loops** that iteratively search and synthesize
- **Grounded generation** that stays faithful to retrieved context

In [ ]:
import requests

BASE_URL = "https://api.contextual.ai/v1"
HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

# Agent Composer YAML — this replaces our entire agents/ directory
acl_yaml = """
version: 0.1
inputs:
  query: str

outputs:
  response: str

nodes:
  create_message_history:
    type: CreateMessageHistoryStep
    input_mapping:
      query: __inputs__#query

  research:
    type: AgenticResearchStep
    ui_stream_types:
      retrievals: true
    config:
      tools_config:
        - name: search_docs
          description: |
            Search the datastore using hybrid semantic and lexical search.
          step_config:
            type: SearchUnstructuredDataStep
            config:
              top_k: 50
              lexical_alpha: 0.1
              semantic_alpha: 0.9
              reranker: "ctxl-rerank-v2-instruct-multilingual-FP8"
              rerank_top_k: 12
              reranker_score_filter_threshold: 0.2

      agent_config:
        agent_loop:
          num_turns: 10
          parallel_tool_calls: false
          identity_guidelines_prompt: |
            You are a retrieval-augmented research assistant. Provide factual,
            grounded answers based on retrieved documents.
          research_guidelines_prompt: |
            Search the datastore comprehensively. Use breadth-then-depth
            retrieval. Avoid redundant searches.

    input_mapping:
      message_history: create_message_history#message_history

  generate:
    type: GenerateFromResearchStep
    ui_stream_types:
      generation: true
    config:
      identity_guidelines_prompt: |
        You are a retrieval-augmented assistant. Provide factual, grounded
        answers based on retrieved information.
      response_guidelines_prompt: |
        - Output concise Markdown with headings and bullets
        - Start immediately with the answer
        - If information is missing, state what is absent

    input_mapping:
      message_history: create_message_history#message_history
      research: research#research

  __outputs__:
    type: output
    input_mapping:
      response: generate#response
""".strip()

print("Agent Composer YAML defined")
print(f"Workflow: CreateMessageHistory -> AgenticResearch -> GenerateFromResearch")

In [ ]:
agent_name = "ODSC_Workshop_Agent"

payload = {
    "name": agent_name,
    "description": "Agentic RAG demo — replaces manual orchestration with Agent Composer",
    "system_prompt": (
        "You are a research assistant that answers questions using retrieved documents. "
        "Always ground your answers in the source material."
    ),
    "suggested_queries": [
        "Summarize the key themes across the uploaded documents",
        "What are the main terms and conditions?",
        "Compare the financial terms across different contracts",
    ],
    "datastore_ids": [datastore_id],
    "agent_configs": {
        "acl_config": {
            "acl_active": True,
            "acl_yaml": acl_yaml,
        }
    },
}

resp = requests.post(f"{BASE_URL}/agents", headers=HEADERS, json=payload)
resp.raise_for_status()
agent = resp.json()

agent_id = agent["id"]
print(f"Created agent: {agent_id}")

## Step 4: Query the Agent

One API call replaces the entire pipeline we built: intent classification, slot filling, query decomposition, dynamic filtering, retrieval, and generation. The agent handles it all — including multi-turn research loops where it iteratively searches and refines.

In [ ]:
query = "What are the main topics covered in the documents?"

payload = {
    "messages": [{"role": "user", "content": query}],
    "stream": False,
}

resp = requests.post(
    f"{BASE_URL}/agents/{agent_id}/query/acl",
    headers=HEADERS,
    json=payload,
)
resp.raise_for_status()
out = resp.json()

print(f"Query: {query}\n")
print(out["message"]["content"])

In [ ]:
# Try a complex, multi-part query — the kind we needed query decomposition for
complex_query = (
    "Compare the salary terms in the employment contracts with the payment "
    "schedules in the service agreements. Which contracts have the most "
    "favorable termination clauses?"
)

payload = {
    "messages": [{"role": "user", "content": complex_query}],
    "stream": False,
}

resp = requests.post(
    f"{BASE_URL}/agents/{agent_id}/query/acl",
    headers=HEADERS,
    json=payload,
)
resp.raise_for_status()
out = resp.json()

print(f"Query: {complex_query}\n")
print(out["message"]["content"])

## Built-In Evaluation: Groundedness and Feedback

Remember Part 4, where we discussed MRR, NDCG, and production failure modes? Contextual AI bakes evaluation into the platform:

- **Groundedness scores** — Measures how faithful the response is to the retrieved context (prevents hallucination)
- **Feedback API** — Track user satisfaction and agent performance over time
- **Metrics API** — Usage data and feedback aggregation for continuous improvement
- **Reranking** — Built-in reranker ensures the best results surface to the top

No custom evaluation pipelines needed.

## Key Takeaways

**What we just did:**
1. Created a managed datastore (replaced Docker + Qdrant setup)
2. Ingested documents with automatic parsing and embedding (replaced manual batch import)
3. Defined an agentic workflow in YAML (replaced our entire `agents/` directory)
4. Queried with a single API call (replaced the full orchestration pipeline)

**When to build from scratch (what we did in Parts 1–4):**
- You need full control over every layer of the stack
- You're building a deeply custom retrieval or agent architecture
- You need to understand how each component works (education, research)

**When to use a managed platform (Contextual AI):**
- You need to ship to production quickly
- You want built-in grounding, evaluation, and observability
- You need enterprise features: data connectors, RBAC, entitlements
- You want to focus on the domain problem, not the infrastructure

**The best approach:** Understand the fundamentals (this workshop), then decide where to invest your engineering effort vs. where to leverage managed infrastructure.

---

Learn more at [contextual.ai](https://contextual.ai) | [Documentation](https://docs.contextual.ai)